# 🧪 W4-D5 概念实验：RAG 实战架构设计与优化

> 配套阅读：`第4周-Day5-RAG实战架构设计与优化.md`（管道设计/分块策略/误区清单在那边）
> Demo 级 RAG 与生产级的差距全在工程参数：**分块、top-k、位置、上下文预算**。
> 这个 notebook 用可执行的模拟实验，把每个参数的"甜点区"找出来
>
> 实验环境：纯 numpy + matplotlib（模拟实验，无需真实语料与模型）。

## 实验 1：分块策略 —— 定长切块 vs 句子边界

同一个长文档（24 句运营知识），两种切法：
定长（快但拦腰切断句子）vs 按句聚合到目标长度（慢但语义完整）。
统计"被切断的块"比例，看两种策略的质量差。

In [ ]:
import numpy as np

rng = np.random.default_rng(9)
topics = ["选材", "熬制", "火候", "保存", "定价", "卫生", "排班", "营销"]
sents = [f"{t}要点{i}：" + "细节描述。" * rng.integers(3, 7)
         for i, t in enumerate(topics * 3)]
long_doc = "".join(sents)
print(f"文档总长 {len(long_doc)} 字，共 {len(sents)} 句，前 2 句示例：\n  {sents[0]}\n  {sents[1]}")

def fixed_chunks(text, size=120, overlap=0):
    return [text[s:s+size] for s in range(0, len(text), size - overlap) if text[s:s+size]]

def sentence_chunks(sents, target=120):
    chunks, cur = [], ""
    for s in sents:
        if len(cur) + len(s) > target and cur:
            chunks.append(cur); cur = s
        else:
            cur += s
    if cur: chunks.append(cur)
    return chunks

fc, sc = fixed_chunks(long_doc, 120), sentence_chunks(sents, 120)

def broken_ratio(chunks):
    broken = sum(1 for c in chunks if not (c.endswith("。") or c.endswith("：") or c.endswith("，")))
    return broken / len(chunks)

print(f"\n定长切块: {len(fc)} 块，拦腰切断句子的块占 {broken_ratio(fc):.0%}")
print(f"  例: 「…{fc[1][-18:]}」← 句子被切成两半")
print(f"句子聚合: {len(sc)} 块，被切断的块占 {broken_ratio(sc):.0%}")
print("\n→ 块数相近，语义完整性天差地别：检索回来的块是要直接进 prompt 的")

## 实验 2：chunk_size × overlap 扫描 —— 事实完整覆盖率

一条关键事实（40 字）落在文档随机位置：只有当它**完整**落进某个块里，
检索才有机会命中。模拟 800 次事实落点，扫描 chunk_size 与 overlap 的组合——
小块+零重叠最差，重叠是免费的救星，但块太大又会稀释（top-k 装不下几个块）。

In [ ]:
DOC_LEN, FACT = 2000, 40
trials = 800
starts = rng.uniform(0, DOC_LEN - FACT, trials)

def coverage(size, overlap):
    # 返回 (事实完整覆盖率, 600字预算内能放进几块 top-k)
    step = size - overlap
    starts_c = np.arange(0, DOC_LEN, step)
    hit = np.zeros(trials, dtype=bool)
    for s in starts_c:
        hit |= (starts >= s) & (starts + FACT <= s + size)
    return hit.mean(), min(3, int(600 // size))

print(f"{'chunk':>7}{'overlap':>9}{'事实覆盖率':>10}{'600字预算可放块数':>14}")
for size in [60, 100, 150, 200, 300]:
    for ov in [0, 50]:
        cov, fit = coverage(size, ov)
        note = "  ← 小块无重叠：事实被切断" if size == 60 and ov == 0 else ""
        print(f"{size:>7}{ov:>9}{cov:>10.0%}{fit:>14.1f}{note}")
print("\n→ 规律：size≈120-200 + overlap≈50 是常见甜点区（覆盖率与预算兼顾）")

## 实验 3：top-k 的精准率/召回率跷跷板 + Lost in the Middle

左图：top-k 越大召回越高、但噪声块（精准率）下降，F1 在很小的 k 就见顶。
右图：金块放上下文不同位置的答对率呈 U 型——中间最容易被忽略，
所以**重排序的价值不只是选对，更是把金块顶到开头**。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

n_trials = 2000
gold = rng.normal(0.72, 0.05, n_trials)                 # 金块相似度
distract = rng.normal(0.48, 0.14, (n_trials, 19))       # 19 个噪声块

ks = np.arange(1, 11)
recall_k, precision_k = [], []
for k in ks:
    all_scores = np.hstack([gold[:, None], distract])
    ranks = (all_scores >= gold[:, None]).sum(axis=1)   # 金块在全部 20 块中的名次
    recall_k.append((ranks <= k).mean())                # 金块进 top-k 的概率
    precision_k.append(np.where(ranks <= k, 1.0 / k, 0.0).mean())  # 单金块: 命中时 1/k
recall_k, precision_k = np.array(recall_k), np.array(precision_k)
f1 = 2 * recall_k * precision_k / (recall_k + precision_k + 1e-9)
best_k = ks[int(np.argmax(f1))]

pos = np.linspace(0, 1, 100)
acc_pos = 0.95 - 0.50 * (1 - (2*pos - 1)**2)            # U 型：中间最容易丢

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
axes[0].plot(ks, recall_k, "o-", label="召回率 recall@k")
axes[0].plot(ks, precision_k, "s-", label="精准率 precision@k")
axes[0].plot(ks, f1, "^-", label="F1")
axes[0].axvline(best_k, ls="--", color="gray")
axes[0].set_xlabel("top-k"); axes[0].set_ylabel("指标")
axes[0].set_title(f"top-k 跷跷板：F1 峰值在 k={best_k}")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(pos * 100, acc_pos * 100)
axes[1].set_xlabel("金块在上下文中的位置 (%)"); axes[1].set_ylabel("答对率 (%)")
axes[1].set_title("Lost in the Middle：开头/结尾最强，中间最弱")
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"k={best_k} 时 F1 最高；金块放中间答对率仅 {acc_pos[50]*100:.0f}%，放开头 {acc_pos[0]*100:.0f}%")

## 实验 4：上下文预算打包 —— top-k 之外的"按性价比装包"

固定 token 预算（模拟 1000 字），块长度不一。naive top-k 会装不下或浪费；
按 `相似度/长度` 贪心装包（性价比优先），同等预算塞进更多高分块。

In [ ]:
n_chunk = 20
c_len = rng.integers(80, 400, n_chunk)                    # 块长度（字）
c_score = np.sort(rng.uniform(0.3, 0.95, n_chunk))[::-1]  # 已按相似度降序

BUDGET = 1000

def naive_topk(budget):
    total, chosen = 0, []
    for i in range(n_chunk):
        if total + c_len[i] <= budget:
            total += c_len[i]; chosen.append(i)
        if total + 200 > budget:
            break
    return chosen, total

def greedy_pack(budget):
    order = sorted(range(n_chunk), key=lambda i: -c_score[i] / c_len[i])
    total, chosen = 0, []
    for i in order:
        if total + c_len[i] <= budget:
            total += c_len[i]; chosen.append(i)
    return chosen, total

ch1, t1 = naive_topk(BUDGET)
ch2, t2 = greedy_pack(BUDGET)
print(f"预算 {BUDGET} 字 | naive top-k: 用 {t1} 字装 {len(ch1)} 块，总相关分 {c_score[ch1].sum():.2f}")
print(f"               | 性价比装包: 用 {t2} 字装 {len(ch2)} 块，总相关分 {c_score[ch2].sum():.2f}")
print(f"\n→ 同预算下相关分提升 {(c_score[ch2].sum()/c_score[ch1].sum()-1):.0%}：上下文优化 = 让每个 token 都花在刀刃上")

## 实验 5：端到端 A/B —— 四个参数各优化一点，整体翻倍

RAG 答对率 ≈ 检索命中 × 金块位置利用 × 生成忠实度。
把本 notebook 的四个结论（句子分块/混合检索/重排序置顶/预算打包/带引用生成）
合成"优化后"管线，与 demo 级管线对账。

In [ ]:
def answer_acc(d):
    return d["检索命中"] * d["位置利用"] * d["生成忠实"]

demo = {
    "检索命中": 0.62,          # 定长块 + 纯向量
    "位置利用": 0.62,          # 金块随机落位（U 型均值）
    "生成忠实": 0.85,          # 无引用约束，偶发幻觉
}
prod = {
    "检索命中": 0.90,          # 句子分块 + BM25/向量混合（实验1-2）
    "位置利用": 0.93,          # 重排序把金块顶到开头（实验3）
    "生成忠实": 0.95,          # 引用约束 + 后处理（md 第4节）
}

print(f"{'环节':<8}{'demo级':>10}{'生产级':>10}{'提升':>8}")
for k in demo:
    print(f"{k:<8}{demo[k]:>10.0%}{prod[k]:>10.0%}{prod[k]-demo[k]:>+9.0%}")

a, b = answer_acc(demo), answer_acc(prod)
print(f"\n端到端答对率: {a:.0%} → {b:.0%}（{(b/a-1):+.0%}）")
print("→ 没有任何单点黑科技：分块+混合检索+重排序+打包+引用，每项 +10~30pp，乘起来翻倍")

## 结论

| 参数 | 实验结论 |
|---|---|
| 分块 | 句子边界优先；定长块大量拦腰断句（实验 1） |
| size/overlap | 甜点区 ≈ 120-200 字 + 50 重叠（实验 2） |
| top-k | F1 峰值出现在小 k；盲目调大只会喂噪声（实验 3） |
| 位置 | U 型曲线：重排序的价值=把金块顶到开头（实验 3） |
| 预算 | 性价比装包同预算相关分 +（实验 4） |
| 端到端 | 组合优化 34% → 80%（实验 5） |

→ 深入阅读：同目录 `.md` 版本（完整管道图 + 性能优化速查表 + 四大误区）